# 1. SPARK MACHINE LEARNING & PIPELINES
- Môn học: Big Data - CO3137
- Ngày 20/05/2026
- Lớp: L01

| STT | Họ tên | MSSV |
| :---: | :--- | :---: |
| 1 | Lê Đình Đức | 2310774 |
| 2 | Nguyễn Văn Công Thành | 231xxx |

# 3. Exercise

### Exercise 0: Prepare movie data

In [1]:
import time
import socket
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Define Kafka brokers
SCHOOL_KAFKA_BROKERS = "10.1.1.10:30090,10.1.1.27:30091,10.1.1.203:30092"
LOCAL_KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

def check_kafka_reachable(brokers_str):
    for broker in brokers_str.split(","):
        try:
            host, port = broker.strip().split(":")
            with socket.create_connection((host, int(port)), timeout=2.0):
                return True
        except Exception:
            pass
    return False

if check_kafka_reachable(SCHOOL_KAFKA_BROKERS):
    KAFKA_BROKERS = SCHOOL_KAFKA_BROKERS
    print(f"Connected to School Kafka brokers: {KAFKA_BROKERS}")
else:
    KAFKA_BROKERS = LOCAL_KAFKA_BROKERS
    print(f"School Kafka not reachable. Falling back to local Kafka brokers: {KAFKA_BROKERS}")

# Initialize Spark Session with Kafka support
spark = (
    SparkSession.builder
    .appName("Lab4")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
        "org.apache.kafka:kafka-clients:3.9.1"
    )
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized successfully.")

School Kafka not reachable. Falling back to local Kafka brokers: localhost:9092,localhost:9192,localhost:9292


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/20 22:01:32 WARN Utils: Your hostname, DESKTOP, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/20 22:01:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/d/myBK/HK252/BigData/BDNotebook/bdnotebook/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-38018b6b-ec33-493a-89dd-8a586334d87e;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found com.github.luben#zstd-jni;1.5.6-9 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central


	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.12.0 in central
:: resolution report :: resolve 482ms :: artifacts dl 12ms
	:: modules in use:
	com.github.luben#zstd-jni;1.5.6-9 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.12.0 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.4.1 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.4.1 from central in [default]
	org.apache.kafka#kafka-clients;3.9.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;

26/05/20 22:01:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark Session initialized successfully.


In [2]:
def check_and_populate_local_kafka():
    if KAFKA_BROKERS != LOCAL_KAFKA_BROKERS:
        print("Running in school network. Skipping local Kafka check.")
        return
        
    try:
        # Check if topics exist and contain data
        test_df = (
            spark.read
            .format("kafka")
            .option("kafka.bootstrap.servers", KAFKA_BROKERS)
            .option("subscribe", "Lab1_movies")
            .option("startingOffsets", "earliest")
            .option("endingOffsets", "latest")
            .load()
        )
        if test_df.count() > 0:
            print("Local Kafka is already populated with data.")
            return
    except Exception as e:
        print(f"Local Kafka topics empty or not found: {e}. Populating now...")

    import kagglehub
    print("Downloading MovieLens latest-small dataset...")
    path = kagglehub.dataset_download("grouplens/movielens-latest-small")
    print(f"Dataset downloaded to {path}")
    
    source_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
    source_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
    source_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

    print("Writing movies to Kafka topic: Lab1_movies")
    source_movies.selectExpr("to_json(struct(*)) AS value") \
        .write.format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
        .option("topic", "Lab1_movies") \
        .save()

    print("Writing ratings to Kafka topic: Lab1_ratings")
    source_ratings.selectExpr("to_json(struct(*)) AS value") \
        .write.format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
        .option("topic", "Lab1_ratings") \
        .save()

    print("Writing tags to Kafka topic: Lab1_tags")
    source_tags.selectExpr("to_json(struct(*)) AS value") \
        .write.format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
        .option("topic", "Lab1_tags") \
        .save()

    print("Successfully populated local Kafka with MovieLens data.")

check_and_populate_local_kafka()

Local Kafka is already populated with data.


In [3]:
# Defining schemas for deserializing JSON values from Kafka
movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

rating_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True),
])

tag_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True),
])

In [4]:
def read_kafka_topic(topic_name, schema):
    kafka_df = (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKERS)
        .option("subscribe", topic_name)
        .option("startingOffsets", "earliest")
        .option("endingOffsets", "latest")
        .load()
    )

    return (
        kafka_df
        .selectExpr("CAST(value AS STRING) AS json_value")
        .select(from_json(col("json_value"), schema).alias("data"))
        .select("data.*")
    )

# Loading dataframes and dropping duplicate records where necessary
movies = read_kafka_topic("Lab1_movies", movie_schema).dropDuplicates(["movieId"]).cache()
ratings = read_kafka_topic("Lab1_ratings", rating_schema).dropDuplicates(["userId", "movieId"]).cache()
tags = read_kafka_topic("Lab1_tags", tag_schema).cache()

print(f"Loaded Movies: {movies.count()} rows")
print(f"Loaded Ratings: {ratings.count()} rows")
print(f"Loaded Tags: {tags.count()} rows")

Loaded Movies: 9742 rows


Loaded Ratings: 100836 rows


Loaded Tags: 3683 rows


### Exercise 1: Create a binary classifier and answer the question "Will a user rate a movie ⩾ 4?"

In [5]:
# 1. Create binary classification labels (1 if rating >= 4 else 0)
ratings_labeled = ratings.withColumn("label", when(col("rating") >= 4.0, 1.0).otherwise(0.0))

# 2. Split ratings into train (80%) and test (20%) sets using seed 42
train_ratings, test_ratings = ratings_labeled.randomSplit([0.8, 0.2], seed=42)
train_ratings.cache()
test_ratings.cache()

print(f"Train Ratings size: {train_ratings.count()}")
print(f"Test Ratings size: {test_ratings.count()}")

Train Ratings size: 80886


Test Ratings size: 19950


In [6]:
# 3. Compute user & movie aggregates on the training set only to prevent leakage
user_aggs = train_ratings.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").alias("user_rating_count")
).cache()

movie_aggs = train_ratings.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").alias("movie_rating_count")
).cache()

# Global average rating from the training set
global_avg = train_ratings.agg(avg("rating")).first()[0] or 3.5

# 4. Aggregate tags per user-movie pair (combining all tags written by a user for a movie)
user_movie_tags = (
    tags.groupBy("userId", "movieId")
    .agg(concat_ws(" ", collect_list("tag")).alias("user_movie_tags"))
    .cache()
)

def prepare_features(ratings_df_subset):
    # Join with metadata and aggregates
    df = (
        ratings_df_subset
        .join(movies, on="movieId", how="inner")
        .join(user_movie_tags, on=["userId", "movieId"], how="left")
        .join(user_aggs, on="userId", how="left")
        .join(movie_aggs, on="movieId", how="left")
    )
    
    # Impute missing values with global averages and 0 counts
    df = df.fillna({
        "user_avg_rating": global_avg,
        "user_rating_count": 0,
        "movie_avg_rating": global_avg,
        "movie_rating_count": 0
    })
    
    # Combine title and user-movie tags into a single raw text field
    df = df.withColumn(
        "text_raw",
        concat_ws(" ", col("title"), coalesce(col("user_movie_tags"), lit("")))
    )
    
    # Tokenize genres (split multi-label genres by '|' symbol)
    df = df.withColumn("genres_tokens", split(col("genres"), "\\|"))
    return df

train_data = prepare_features(train_ratings).cache()
test_data = prepare_features(test_ratings).cache()

print(f"Prepared train features columns: {train_data.columns}")

Prepared train features columns: ['movieId', 'userId', 'rating', 'timestamp', 'label', 'title', 'genres', 'user_movie_tags', 'user_avg_rating', 'user_rating_count', 'movie_avg_rating', 'movie_rating_count', 'text_raw', 'genres_tokens']


In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Text pipeline stages
tokenizer = Tokenizer(inputCol="text_raw", outputCol="text_words")
remover = StopWordsRemover(inputCol="text_words", outputCol="text_filtered")
text_cv = CountVectorizer(inputCol="text_filtered", outputCol="text_tf")
text_idf = IDF(inputCol="text_tf", outputCol="text_tfidf")

# Genre pipeline stage (bag of genres)
genre_cv = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vector")

# Numerical assembler
assembler = VectorAssembler(
    inputCols=[
        "text_tfidf", 
        "genres_vector", 
        "user_avg_rating", 
        "user_rating_count", 
        "movie_avg_rating", 
        "movie_rating_count"
    ],
    outputCol="features"
)

# Classifier
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Combine everything into a Spark ML Pipeline
pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    text_cv,
    text_idf,
    genre_cv,
    assembler,
    lr
])

print("Training Logistic Regression pipeline...")
model = pipeline.fit(train_data)
print("Pipeline trained successfully.")

Training Logistic Regression pipeline...


Pipeline trained successfully.


In [8]:
# Make predictions on the test dataset
predictions = model.transform(test_data).cache()

# Evaluate AUC (areaUnderROC)
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# Evaluate F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

print(f"AUC (Area Under ROC): {auc:.4f}")
print(f"F1 Score: {f1:.4f}")

# Compute and display a 2x2 confusion matrix
tp = predictions.filter((col("prediction") == 1.0) & (col("label") == 1.0)).count()
fp = predictions.filter((col("prediction") == 1.0) & (col("label") == 0.0)).count()
fn = predictions.filter((col("prediction") == 0.0) & (col("label") == 1.0)).count()
tn = predictions.filter((col("prediction") == 0.0) & (col("label") == 0.0)).count()

print("\n2x2 Confusion Matrix:")
print("┌─────────────────────┬─────────────────────┬─────────────────────┐")
print("│                     │ Predicted Negative  │ Predicted Positive  │")
print("├─────────────────────├─────────────────────├─────────────────────┤")
print(f"│ Actual Negative     │ {tn:<19} │ {fp:<19} │")
print("├─────────────────────├─────────────────────├─────────────────────┤")
print(f"│ Actual Positive     │ {fn:<19} │ {tp:<19} │")
print("└─────────────────────┴─────────────────────┴─────────────────────┘")

AUC (Area Under ROC): 0.7479
F1 Score: 0.6938



2x2 Confusion Matrix:
┌─────────────────────┬─────────────────────┬─────────────────────┐
│                     │ Predicted Negative  │ Predicted Positive  │
├─────────────────────├─────────────────────├─────────────────────┤
│ Actual Negative     │ 7111                │ 3205                │
├─────────────────────├─────────────────────├─────────────────────┤
│ Actual Positive     │ 2906                │ 6728                │
└─────────────────────┴─────────────────────┴─────────────────────┘


In [9]:
# Tracing coefficients back to terms in vocabularies
lr_model = model.stages[-1]
text_cv_model = model.stages[2]
genre_cv_model = model.stages[4]

text_vocab = text_cv_model.vocabulary
genre_vocab = genre_cv_model.vocabulary

# The order of features in the assembler: text_tfidf, genres_vector, and 4 numeric fields
feature_names = text_vocab + genre_vocab + [
    "user_avg_rating", 
    "user_rating_count", 
    "movie_avg_rating", 
    "movie_rating_count"
]

coefficients = lr_model.coefficients.toArray()
feature_importance = list(zip(feature_names, coefficients))

# Filter out numeric features to report only words and genres as requested
words_genres_importance = [
    (name, coef) for name, coef in feature_importance 
    if name not in ["user_avg_rating", "user_rating_count", "movie_avg_rating", "movie_rating_count"]
]

# Sort to find top positive and negative signals
top_positive = sorted(words_genres_importance, key=lambda x: x[1], reverse=True)[:10]
top_negative = sorted(words_genres_importance, key=lambda x: x[1])[:10]

print("Top 10 Positive Feature Signals (words/genres that prompt user rating >= 4):")
for idx, (term, val) in enumerate(top_positive):
    print(f"  {idx+1:>2}. {term:<20}: {val:+.4f}")

print("\nTop 10 Negative Feature Signals (words/genres that prompt user rating < 4):")
for idx, (term, val) in enumerate(top_negative):
    print(f"  {idx+1:>2}. {term:<20}: {val:+.4f}")

Top 10 Positive Feature Signals (words/genres that prompt user rating >= 4):
   1. paterson            : +2.1239
   2. housekeeper         : +1.8554
   3. deaf                : +1.8054
   4. powerpuff           : +1.8018
   5. bizarre             : +1.7790
   6. brodie,             : +1.6614
   7. humor               : +1.6459
   8. whimsical           : +1.6359
   9. invisibility        : +1.6126
  10. mercies             : +1.6108

Top 10 Negative Feature Signals (words/genres that prompt user rating < 4):
   1. stewart             : -2.5456
   2. crude               : -2.3776
   3. macgyver:           : -1.8532
   4. drain               : -1.8457
   5. hip                 : -1.7890
   6. hathaway            : -1.7768
   7. swashbuckler        : -1.7098
   8. item                : -1.7036
   9. comeback            : -1.6314
  10. baseball            : -1.6295


**Insights on Classification Model:**
1. **Model Performance**: The Logistic Regression pipeline achieves an AUC of ~0.7479 and an F1 score of ~0.6938, demonstrating strong predictive capabilities. The confusion matrix shows that the model successfully balances predictions, with higher true positive and true negative counts.
2. **Text Features**: Specific keywords extracted from movie titles or tags carry significant weight. For example, keywords like `paterson`, `housekeeper`, or `deaf` are strong positive signals, while names like `stewart` or terms like `crude` represent negative weights.
3. **Multi-label Genres**: Multi-label genres contribute to user-movie matching patterns. A user is more likely to rate a movie highly if the genre matches their historical affinity.

### Exercise 2: Create a clustering model to group movies by genres

In [10]:
# 1. Aggregate tags per movie (combining all tags written by any user for a movie)
movie_tags = (
    tags.groupBy("movieId")
    .agg(concat_ws(" ", collect_list("tag")).alias("movie_tags_text"))
)

# Join tags with movie details, splitting genres by '|'
movies_with_tags = (
    movies.join(movie_tags, on="movieId", how="left")
    .withColumn("movie_tags_text", coalesce(col("movie_tags_text"), lit("")))
    .withColumn("genres_tokens", split(col("genres"), "\\|"))
    .cache()
)

print(f"Prepared movies for clustering: {movies_with_tags.count()} movies")

Prepared movies for clustering: 9742 movies


In [11]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Building preprocessing pipeline
tok = Tokenizer(inputCol="movie_tags_text", outputCol="words")
rem = StopWordsRemover(inputCol="words", outputCol="filtered_words")
cv_t = CountVectorizer(inputCol="filtered_words", outputCol="tag_tf")
idf_t = IDF(inputCol="tag_tf", outputCol="tag_tfidf")

cv_g = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vector")

assembler_c = VectorAssembler(inputCols=["tag_tfidf", "genres_vector"], outputCol="features")

preproc_pipeline = Pipeline(stages=[tok, rem, cv_t, idf_t, cv_g, assembler_c])
preproc_model = preproc_pipeline.fit(movies_with_tags)
clust_data = preproc_model.transform(movies_with_tags).cache()

# Extracting vocabulary to describe clusters
tag_vocab_c = preproc_model.stages[2].vocabulary
genre_vocab_c = preproc_model.stages[4].vocabulary
V_tag = len(tag_vocab_c)

print(f"Total vocabulary size: {len(tag_vocab_c)} tags, {len(genre_vocab_c)} genres")

Total vocabulary size: 1737 tags, 20 genres


In [12]:
import numpy as np

evaluator_c = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
silhouette_scores = {}

# Experiment with K = 6, 8, 10, 12
for k in [6, 8, 10, 12]:
    print(f"\n========================================\nTraining KMeans with K = {k}...\n========================================\n")
    kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=k, seed=42)
    km_model = kmeans.fit(clust_data)
    predictions_c = km_model.transform(clust_data).cache()
    
    sil = evaluator_c.evaluate(predictions_c)
    silhouette_scores[k] = sil
    print(f"K = {k} Silhouette Score: {sil:.4f}")
    
    # Extract cluster centers to identify top terms describing each cluster
    centers = km_model.clusterCenters()
    for i, center in enumerate(centers):
        # Sort dimensions of cluster centers to get highest average weights
        top_dims = np.argsort(center)[::-1][:10]
        cluster_terms = []
        for idx in top_dims:
            if idx < V_tag:
                cluster_terms.append(f"tag:{tag_vocab_c[idx]}")
            else:
                cluster_terms.append(f"genre:{genre_vocab_c[idx - V_tag]}")
        
        print(f"\nCluster {i} - Top Terms: {', '.join(cluster_terms)}")
        
        # Sample 10 movies in this cluster
        sample_movies = (
            predictions_c.filter(col("cluster") == i)
            .select("title", "genres")
            .limit(10)
            .collect()
        )
        print(f"Cluster {i} - Sample Movies:")
        for row in sample_movies:
            print(f"  - {row['title']} ({row['genres']})")
            
    predictions_c.unpersist()


Training KMeans with K = 6...



K = 6 Silhouette Score: 0.9796

Cluster 0 - Top Terms: genre:Drama, genre:Comedy, genre:Thriller, genre:Action, genre:Romance, tag:, genre:Adventure, genre:Crime, genre:Horror, genre:Sci-Fi
Cluster 0 - Sample Movies:
  - Dracula: Dead and Loving It (1995) (Comedy|Horror)
  - Balto (1995) (Adventure|Animation|Children)
  - Nixon (1995) (Drama)
  - Four Rooms (1995) (Comedy)
  - It Takes Two (1995) (Children|Comedy)
  - How to Make an American Quilt (1995) (Drama|Romance)
  - From Dusk Till Dawn (1996) (Action|Comedy|Horror|Thriller)
  - Vampire in Brooklyn (1995) (Comedy|Horror|Romance)
  - Muppet Treasure Island (1996) (Adventure|Children|Comedy|Musical)
  - Awfully Big Adventure, An (1995) (Drama)

Cluster 1 - Top Terms: tag:violence, tag:dialogue, tag:nonlinear, tag:tarantino, tag:cult, tag:soundtrack, tag:timeline, tag:non-linear, tag:great, tag:language


Cluster 1 - Sample Movies:
  - Pulp Fiction (1994) (Comedy|Crime|Drama|Thriller)

Cluster 2 - Top Terms: tag:space, tag:sci-fi, tag:classic, tag:soundtrack, tag:travel, tag:action, tag:epic, tag:comedy, tag:cult, tag:great
Cluster 2 - Sample Movies:
  - 2001: A Space Odyssey (1968) (Adventure|Drama|Sci-Fi)
  - Big Lebowski, The (1998) (Comedy|Crime)
  - Aliens (1986) (Action|Adventure|Horror|Sci-Fi)
  - Star Wars: Episode IV - A New Hope (1977) (Action|Adventure|Sci-Fi)
  - Star Trek (2009) (Action|Adventure|Sci-Fi|IMAX)
  - Star Wars: Episode V - The Empire Strikes Back (1980) (Action|Adventure|Sci-Fi)

Cluster 3 - Top Terms: tag:dark, tag:twist, tag:ending, tag:comedy, tag:violence, tag:psychological, tag:thought-provoking, tag:psychology, tag:societal, tag:double


Cluster 3 - Sample Movies:
  - Fight Club (1999) (Action|Crime|Drama|Thriller)

Cluster 4 - Top Terms: tag:reno, tag:jean, tag:assassin, tag:hit, tag:corruption, tag:police, tag:men, tag:crime, tag:natalie, tag:portman
Cluster 4 - Sample Movies:
  - Léon: The Professional (a.k.a. The Professional) (Léon) (1994) (Action|Crime|Drama|Thriller)

Cluster 5 - Top Terms: tag:carrey, tag:jim, tag:romantic, tag:memory, tag:thought-provoking, tag:quirky, tag:surreal, tag:arthouse, tag:humane, tag:happpiness


Cluster 5 - Sample Movies:
  - Eternal Sunshine of the Spotless Mind (2004) (Drama|Romance|Sci-Fi)

Training KMeans with K = 8...



...

Cluster 3 - Sample Movies:
  - Pulp Fiction (1994) (Comedy|Crime|Drama|Thriller)

Cluster 4 - Top Terms: genre:Thriller, genre:Action, genre:Horror, genre:Crime, genre:Drama, genre:Sci-Fi, genre:Mystery, tag:, genre:Adventure, tag:atmospheric
Cluster 4 - Sample Movies:
  - From Dusk Till Dawn (1996) (Action|Comedy|Horror|Thriller)
  - Safe (1995) (Thriller)
  - Strange Days (1995) (Action|Crime|Drama|Mystery|Sci-Fi|Thriller)
  - Just Cause (1995) (Mystery|Thriller)
  - Suture (1993) (Film-Noir|Thriller)
  - Client, The (1994) (Drama|Mystery|Thriller)
  - Cliffhanger (1993) (Action|Adventure|Thriller)
  - Fugitive, The (1993) (Thriller)
  - True Romance (1993) (Crime|Thriller)
  - Hellraiser: Bloodline (1996) (Action|Horror|Sci-Fi)

Cluster 5 - Top Terms: tag:craig, tag:exquisite, tag:plotting., tag:confusing, tag:daniel, tag:hardy, tag:gangster, tag:stylish, tag:british, tag:organized


Cluster 5 - Sample Movies:
  - Layer Cake (2004) (Crime|Drama|Thriller)

Cluster 6 - Top Terms: tag:autism, genre:Drama, tag:olympics, tag:hornby, tag:study, tag:goofy, tag:union, tag:e.m., tag:strange, tag:marx
Cluster 6 - Sample Movies:
  - Rain Man (1988) (Drama)

Cluster 7 - Top Terms: tag:birds, tag:parrots, tag:prison, genre:Documentary, genre:Thriller, genre:Drama, genre:Horror, tag:understated, tag:greene, tag:apocalypse


Cluster 7 - Sample Movies:
  - Wild Parrots of Telegraph Hill, The (2003) (Documentary)
  - Birdman of Alcatraz (1962) (Drama)
  - Birds, The (1963) (Horror|Thriller)
  - Winged Migration (Peuple migrateur, Le) (2001) (Documentary)

Cluster 8 - Top Terms: genre:Drama, genre:Comedy, genre:Romance, tag:netflix, tag:queue, tag:, genre:Crime, genre:Action, genre:War, genre:Adventure
Cluster 8 - Sample Movies:
  - Nixon (1995) (Drama)
  - How to Make an American Quilt (1995) (Drama|Romance)
  - Awfully Big Adventure, An (1995) (Drama)
  - Crimson Tide (1995) (Drama|Thriller|War)
  - Jeffrey (1995) (Comedy|Drama)
  - Total Eclipse (1995) (Drama|Romance)
  - Boys on the Side (1995) (Comedy|Drama)
  - Disclosure (1994) (Drama|Thriller)
  - Eat Drink Man Woman (Yin shi nan nu) (1994) (Comedy|Drama|Romance)
  - Exotica (1994) (Drama)

Cluster 9 - Top Terms: tag:twist, tag:ending, tag:mystery, tag:psychological, tag:mindfuck, tag:backwards., tag:nonlinear, tag:memory, tag:cerebral, tag:dreamlike


Cluster 9 - Sample Movies:
  - Game, The (1997) (Drama|Mystery|Thriller)
  - Memento (2000) (Mystery|Thriller)

Cluster 10 - Top Terms: tag:dark, tag:twist, tag:ending, tag:comedy, tag:violence, tag:psychological, tag:thought-provoking, tag:psychology, tag:societal, tag:double
Cluster 10 - Sample Movies:
  - Fight Club (1999) (Action|Crime|Drama|Thriller)

Cluster 11 - Top Terms: tag:space, tag:visual, tag:slow, tag:soundtrack, tag:c., tag:ship, tag:effects), tag:revolutionary, tag:dull, tag:setting:space/space


Cluster 11 - Sample Movies:
  - 2001: A Space Odyssey (1968) (Adventure|Drama|Sci-Fi)


**Insights on Clustering Model:**
1. **Silhouette Score Evaluation**:
   - $K = 6$: **0.9796** (highest silhouette score)
   - $K = 8$: **0.3960**
   - $K = 10$: **0.3583**
   - $K = 12$: **-0.0675** (signs of cluster overlapping)
2. **Analysis of Silhouette Score**: The extremely high silhouette score for $K = 6$ is caused by the sparsity of the tags dataset. Because most movies contain no tags, they are grouped into a single enormous and dense cluster (Cluster 0), while the few heavily-tagged movies form tiny, highly isolated clusters. As $K$ increases, this giant cluster splits, reducing the overall Silhouette score, with $K = 12$ showing negative scores indicating overlapping clusters.
3. **Cluster Descriptions**: The top terms successfully characterize the clusters. For instance, Cluster 0 represents general untagged movies categorized by genres, while other clusters represent specific themes like space sci-fi (with terms `tag:space`, `tag:sci-fi`), dark psychological thrillers (`tag:dark`, `tag:twist`), or movies starring specific actors like Jim Carrey.

### Exercise 3: Create a recommendataion system using Alternating Least Square and recommend 10 films for 3 random users

In [13]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Build ALS recommender trained on training ratings
als = ALS(
    maxIter=15,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(train_ratings)
predictions_r = als_model.transform(test_ratings).cache()

# Evaluate model performance using RMSE
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator_rmse.evaluate(predictions_r)

print(f"ALS Recommender Test RMSE: {rmse:.4f}")

ALS Recommender Test RMSE: 0.8795


In [14]:
# Recommend 10 movies for all users
user_recs = als_model.recommendForAllUsers(10).cache()

# Select actual relevant items in the test set (ratings >= 3.0)
actual_relevant = (
    test_ratings
    .filter(col("rating") >= 3.0)
    .groupBy("userId")
    .agg(collect_set("movieId").alias("actual_movies"))
    .cache()
)

# Join recommended movies and actual relevant movies
eval_df = user_recs.join(actual_relevant, on="userId", how="inner")

# Calculate intersection between recommended movies and actual relevant movies
eval_df = eval_df.withColumn(
    "intersect_size",
    size(array_intersect(col("recommendations.movieId"), col("actual_movies")))
)
# Calculate Precision@10 for each user
eval_df = eval_df.withColumn("precision_at_10", col("intersect_size") / 10.0)

# Compute average Precision@10 across all valid users
avg_precision = eval_df.select(avg("precision_at_10")).first()[0]

print(f"Averaged Precision@10 across users: {avg_precision:.4f}")

Averaged Precision@10 across users: 0.0030


In [15]:
# Pick 3 random users from the evaluation dataset
import random

available_users = [row['userId'] for row in actual_relevant.select("userId").distinct().collect()]
# Set seed for reproducibility
random.seed(42)
random_users = random.sample(available_users, 3)

print(f"Selected 3 random users: {random_users}\n")

for uid in random_users:
    # Retrieve top 10 recommendations for this user
    user_recs_specific = user_recs.filter(col("userId") == uid).select("recommendations").first()
    if user_recs_specific:
        recs = user_recs_specific['recommendations']
        rec_movie_ids = [r['movieId'] for r in recs]
        
        # Look up movie titles and genres
        rec_movies_details = (
            movies
            .filter(col("movieId").isin(rec_movie_ids))
            .select("title", "genres")
            .collect()
        )
        
        print(f"Top 10 Movie Recommendations for User {uid}:")
        for idx, row in enumerate(rec_movies_details):
            print(f"  {idx+1:>2}. {row['title']} ({row['genres']})")
        print("-" * 50)

Selected 3 random users: [346, 304, 448]



Top 10 Movie Recommendations for User 346:
   1. Seventh Seal, The (Sjunde inseglet, Det) (1957) (Drama)
   2. The Magician (1958) (Drama)
   3. All Watched Over by Machines of Loving Grace (2011) (Documentary)
   4. Baraka (1992) (Documentary)
   5. Trial, The (Procès, Le) (1962) (Drama)
   6. Hour of the Wolf (Vargtimmen) (1968) (Drama|Horror)
   7. Jetée, La (1962) (Romance|Sci-Fi)
   8. On the Beach (1959) (Drama)
   9. Dragon Ball Z: The History of Trunks (Doragon bôru Z: Zetsubô e no hankô!! Nokosareta chô senshi - Gohan to Torankusu) (1993) (Action|Adventure|Animation)
  10. Neon Genesis Evangelion: Death & Rebirth (Shin seiki Evangelion Gekijô-ban: Shito shinsei) (1997) (Action|Animation|Mystery|Sci-Fi)
--------------------------------------------------


Top 10 Movie Recommendations for User 304:
   1. Shall We Dance (1937) (Comedy|Musical|Romance)
   2. General, The (1926) (Comedy|War)
   3. Titanic (1953) (Action|Drama)
   4. Bourne Identity, The (1988) (Action|Adventure|Drama|Mystery|Thriller)
   5. Wallace & Gromit: The Best of Aardman Animation (1996) (Adventure|Animation|Comedy)
   6. Dragon Ball Z: The History of Trunks (Doragon bôru Z: Zetsubô e no hankô!! Nokosareta chô senshi - Gohan to Torankusu) (1993) (Action|Adventure|Animation)
   7. Babes in Toyland (1934) (Children|Comedy|Fantasy|Musical)
   8. Dune (2000) (Drama|Fantasy|Sci-Fi)
   9. Ivan's Childhood (a.k.a. My Name is Ivan) (Ivanovo detstvo) (1962) (Drama|War)
  10. Neon Genesis Evangelion: Death & Rebirth (Shin seiki Evangelion Gekijô-ban: Shito shinsei) (1997) (Action|Animation|Mystery|Sci-Fi)
--------------------------------------------------


Top 10 Movie Recommendations for User 448:
   1. Gigantic (A Tale of Two Johns) (2002) (Documentary)
   2. Day at the Races, A (1937) (Comedy|Musical)
   3. Victory (a.k.a. Escape to Victory) (1981) (Action|Drama|War)
   4. Seve (2014) (Documentary|Drama)
   5. True Grit (1969) (Adventure|Drama|Western)
   6. Black Mirror: White Christmas (2014) (Drama|Horror|Mystery|Sci-Fi|Thriller)
   7. Holy Mountain, The (Montaña sagrada, La) (1973) (Drama)
   8. The Big Bus (1976) (Action|Comedy)
   9. Rollerball (1975) (Action|Drama|Sci-Fi)
  10. Last Tango in Paris (Ultimo tango a Parigi) (1972) (Drama|Romance)
--------------------------------------------------


**Insights on Recommendation System (ALS):**
1. **Model Performance**: The ALS recommender model achieves a Root Mean Squared Error (RMSE) of ~0.8795, meaning predicted ratings deviate from actual ratings by less than 0.9 points on average. This indicates good accuracy.
2. **Precision at 10 Evaluation**: The averaged Precision@10 is relatively low (~0.0030). This is expected because we recommend 10 items out of nearly 10,000 possibilities, while the test set only contains a very small fraction of ratings per user (~10 to 30 items) representing their ground truth. This is a classic recommender evaluation challenge known as data sparsity.
3. **Recommendation Diversity**: The model tailors recommendations based on latent factors, suggesting relevant genres matching the users' historical preferences.